# **算子仿真调优**

本节为算子仿真调优章节，完成本章节内容的学习可以掌握如何使用msProf工具采集仿真性能数据，并分析算子仿真性能瓶颈。我们将按照以下结构，带你学习算子仿真调优流程：
- 环境准备
- 如何使用msProf工具采集仿真性能数据
- 如何分析仿真性能数据
- 针对性优化
- 课后实践

---

## **1环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及正常使用工具。

In [ ]:
!mkdir -p Sources/04.04


import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

---
## **2如何使用msProf工具采集仿真性能数据**

上一节已经介绍了如何使用msProf采集上板性能数据。相对于上板模式，仿真模式多了 `simulator` 参数，并且需要指定仿真的产品型号。对于本节使用的自包含Add算子样例，可在CMake配置阶段通过 `CMAKE_ASC_RUN_MODE=sim` 开启仿真编译，并通过 `CMAKE_ASC_ARCHITECTURES=dav-2201` 指定NPU架构版本。同时，为了让仿真trace能够正常显示源码代码行，需要在运行目录的 `CMakeLists.txt` 中为Ascend C编译添加 `-g`。

所以仿真性能采集分为3步：  

1. 准备测试程序并添加源码行信息编译选项
2. 使用仿真编译参数编译测试程序
3. 采集仿真数据


这里继续使用 `./src/04.03` 中的自包含Add算子测试程序，复制到 `Sources/04.04` 后用于仿真采集。

In [ ]:
# 清理并准备测试程序目录
!rm -rf Sources/04.04
!mkdir -p Sources/04.04

# 复制已准备好的自包含 Add 算子样例
!cp ./src/04.03/add_custom.asc Sources/04.04/add_custom.asc
!cp ./src/04.03/CMakeLists.txt Sources/04.04/CMakeLists.txt


测试程序目录结构为：

```
Sources/04.04/
├── add_custom.asc    // 包含Kernel实现、Host侧调用代码和main函数
└── CMakeLists.txt    // 使用Ascend C CMake工具链编译demo可执行程序
```


复制得到的 `CMakeLists.txt` 来自共享测试程序目录。为了让仿真trace能够显示源码代码行，本节只在运行目录 `Sources/04.04/CMakeLists.txt` 中增加 `-g` 编译选项；仿真编译所需配置仍通过CMake配置命令传入：

```shell
cmake -B build -DCMAKE_ASC_RUN_MODE=sim -DCMAKE_ASC_ARCHITECTURES=dav-2201
```

其中 `CMAKE_ASC_RUN_MODE=sim` 表示开启仿真编译，`CMAKE_ASC_ARCHITECTURES=dav-2201` 表示本节样例使用的NPU架构版本。


In [ ]:
%%writefile Sources/04.04/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    add_custom.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
    $<$<COMPILE_LANGUAGE:ASC>:-g>
)


然后根据仿真产品型号设置仿真相关的环境变量，这里以仿真Atlas A2训练产品为例，应设置环境变量为：

```shell
export LD_LIBRARY_PATH=${ASCEND_TOOLKIT_HOME}/tools/simulator/Ascend910B1/lib:$LD_LIBRARY_PATH
```

设置环境变量后，即可使用 `msprof op simulator` 抓取可执行程序执行的仿真性能，命令为:

```shell
msprof op simulator --output=./output_data ./xxxx
```

如果不设置 `LD_LIBRARY_PATH`，也可通过增加 `--soc-version` 指定要仿真的产品型号，命令为:

```shell
msprof op simulator --soc-version=Ascend910B1 --output=./output_data ./xxxx
```

如果确认核间数据为均匀分布，或者只想获取指定核的仿真数据，可以通过 `--core-id` 来指定核。以采集id为0的核的仿真性能为例，命令为：

```shell
msprof op simulator --soc-version=Ascend910B1 --output=./output_data --core-id=0 ./xxxx
```

让我们执行以下命令编译测试程序，尝试采集id为0的仿真性能数据。


In [ ]:
# 清除可能存在的性能文件
!rm -rf Sources/04.04/prof
# 创建性能文件存放目录
!mkdir -p Sources/04.04/prof
# 编译测试程序
!cd Sources/04.04 && cmake -B build -DCMAKE_ASC_RUN_MODE=sim -DCMAKE_ASC_ARCHITECTURES=dav-2201 && cmake --build build
# 采集仿真性能数据
!msprof op simulator --soc-version=Ascend910B1 --output=./Sources/04.04/prof --core-id=0 ./Sources/04.04/build/demo


命令执行后，Sources/04.04/prof目录下会生成性能数据文件，目录结构如下:

```
OPPROF_20260310020803_BJNTOKUSUZMIESZL/
├── simulator/
│   ├── core0.veccore0/                   // 按照core*.veccore*或core*.cubecore*目录存放各核的数据文件
│   ├── trace.json                       // 全部核或指定核的仿真指令流水图文件
│   └── visualize_data.bin               // 全部核或指定核的仿真指令流水图文件
└── dump/                                  // 存放过程件的文件夹，无需关注
    ├── aicore_binary.o
    ├── object_dump.txt
    └── pc_start_addr.txt
```


执行以下命令查看Sources/04.04/prof查看到采集的性能数据文件。

In [ ]:
!cd Sources/04.04/prof; find . -maxdepth 3 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

---
## **3如何分析仿真性能数据**

已抓取的仿真数据文件trace.json、visualize_data.bin通常不能直接分析性能，这里我们需要借助Chrome浏览器的chrome://tracing打开trace.json或者使用[MindStudio Insight工具](https://www.hiascend.com/document/detail/zh/mindstudio/840/GUI_baseddevelopmenttool/msascendinsightug/Insight_userguide_0027.html)打开trace.json、visualize_data.bin。 

### **3.1 Chrome浏览器打开**
在Chrome浏览器中输入“chrome://tracing”地址，并将通过msprof op simulator生成指令流水图文件（trace.json）拖到空白处打开，键盘上输入快捷键（**W：放大，S：缩小，A：左移，D：右移**）可进行查看看各个流水任务耗时的信息。例如刚刚我们抓取的Add算子样例性能文件打开会如下：  

<img src="./images/trace_info.png"  alt="trace"  width="1920px">    

我们可以比较直观的看到算子整体耗时中MTE2、VECTOR、MET3耗时均很短，而SCALAR运算耗时很长，点击右侧SCALAR可在下方看到该流水任务对应的具体代码行数。
执行以下代码下载我们抓取到的trace.json文件，尝试使用Chrome浏览器查看一下吧。

In [ ]:
import glob,base64;from IPython.display import display,HTML
f=glob.glob("Sources/04.04/prof/OPPROF_*/simulator/trace.json")
if f:
    b64=base64.b64encode(open(f[0],'rb').read()).decode()
    display(HTML(f'<a href="data:application/json;base64,{b64}" download="trace.json" style="color:white;background:#007bff;padding:5px 10px;text-decoration:none;border-radius:3px;">📥 下载</a>'))

也可以直接执行下面代码查看性能文件，该代码简单生成了一个图表模拟了chrome://tracing的展示效果，但是不可以缩放和拖动。

In [ ]:
import matplotlib.pyplot as plt, warnings, json, pandas as pd, os, glob
plt.set_loglevel("error"); warnings.filterwarnings('ignore', category=UserWarning)

BASE_DIR = "Sources/04.04/prof"
try:
    trace_path = f"{sorted(glob.glob(f'{BASE_DIR}/OPPROF*'), key=os.path.getctime)[-1]}/simulator/trace.json"
except:
    trace_path = "Sources/04.04/prof/OPPROF_20260310110753_XXSVYUAHKSEHAZXS/simulator/trace.json"

df = pd.DataFrame([{"name":e.get("name","unknown"),"start":e["ts"]/1e6,"dur":e["dur"]/1e6,"thread":f"Thread-{e.get('tid',0)}"} 
                   for e in json.load(open(trace_path))["traceEvents"] if "ts" in e and "dur" in e])
plt.figure(figsize=(12, 5))

for thread in df["thread"].unique():
    t_df = df[df["thread"] == thread]
    plt.barh(thread, t_df["dur"], left=t_df["start"], label=t_df["name"].iloc[0], alpha=0.7)

thread_names = [th.replace("Thread-", "") for th in df["thread"].unique()]
plt.yticks(ticks=df["thread"].unique(), labels=thread_names)

plt.xlabel("Time (ms)"); plt.title("Trace.json")
plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1)); plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

### **3.2使用MindStudio Insight打开**
若要使用MindStudio Insight进行查看时，需要单独安装MindStudio Insight软件包，具体下载链接请参见[安装与卸载](https://www.hiascend.com/document/detail/zh/mindstudio/840/GUI_baseddevelopmenttool/msascendinsightug/Insight_userguide_0006.html)。  
安装好MindStudio Insight软件包后，我们可以在工具中导入刚刚抓取到的性能数据文件。   

<img src="./images/chose_file.png" alt="trace"  width="700px">

导入后，时间线界面与Chrome浏览器中打开效果一致，如图:   

<img src="./images/timeline.png" alt="trace"  width="700px">

切换到源码界面，我们在设置好对应源码后，即可查看每行代码对应的时钟周期，可以更直观的看到耗时较多的代码或代码块。  

<img src="./images/code_info.png" alt="trace"  width="1920px">

---
## **4针对性优化**

根据trace图或者源码时钟周期我们可以看出耗时主要集中在`printf`打印，所以推测将`printf`打印屏蔽后，算子的性能会有较大提升。这里通过CMake编译定义`ASCENDC_DUMP=0`屏蔽打印，并重新采集仿真性能数据进行对比。


In [ ]:
%%writefile Sources/04.04/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    add_custom.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
    $<$<COMPILE_LANGUAGE:ASC>:-g>
)

target_compile_definitions(demo PRIVATE ASCENDC_DUMP=0)


修改完成后重新部署算子并抓取性能：

In [ ]:
# 清理旧构建目录并重新编译
!rm -rf Sources/04.04/build
!cd Sources/04.04 && cmake -B build -DCMAKE_ASC_RUN_MODE=sim -DCMAKE_ASC_ARCHITECTURES=dav-2201 && cmake --build build

# 清除可能存在的性能文件
!rm -rf Sources/04.04/prof2
# 创建性能文件存放目录
!mkdir -p Sources/04.04/prof2
# 采集屏蔽打印后的仿真性能数据
!msprof op simulator --soc-version=Ascend910B1 --output=./Sources/04.04/prof2 --core-id=0 ./Sources/04.04/build/demo


抓取仿真性能后，使用Chrome浏览器读取性能数据与之前对比，观察是否符合预期，减少了scalar运算耗时。

In [ ]:
import glob,base64;from IPython.display import display,HTML
f=glob.glob("Sources/04.04/prof2/OPPROF_*/simulator/trace.json")
if f:
    b64=base64.b64encode(open(f[0],'rb').read()).decode()
    display(HTML(f'<a href="data:application/json;base64,{b64}" download="trace.json" style="color:white;background:#007bff;padding:5px 10px;text-decoration:none;border-radius:3px;">📥 下载</a>'))

预期如图：  

<img src="./images/no_print_trace.png" alt="prof_no_printf"  width="1920px">


或执行下面代码简单查看

In [ ]:
import matplotlib.pyplot as plt, warnings, json, pandas as pd, os, glob
plt.set_loglevel("error"); warnings.filterwarnings('ignore', category=UserWarning)

BASE_DIR = "Sources/04.04/prof2"
try:
    trace_path = f"{sorted(glob.glob(f'{BASE_DIR}/OPPROF*'), key=os.path.getctime)[-1]}/simulator/trace.json"
except:
    trace_path = "Sources/04.04/prof2/OPPROF_20260310110753_XXSVYUAHKSEHAZXS/simulator/trace.json"

df = pd.DataFrame([{"name":e.get("name","unknown"),"start":e["ts"]/1e6,"dur":e["dur"]/1e6,"thread":f"Thread-{e.get('tid',0)}"} 
                   for e in json.load(open(trace_path))["traceEvents"] if "ts" in e and "dur" in e])
plt.figure(figsize=(12, 5))

for thread in df["thread"].unique():
    t_df = df[df["thread"] == thread]
    plt.barh(thread, t_df["dur"], left=t_df["start"], label=t_df["name"].iloc[0], alpha=0.7)

thread_names = [th.replace("Thread-", "") for th in df["thread"].unique()]
plt.yticks(ticks=df["thread"].unique(), labels=thread_names)

plt.xlabel("Time (ms)"); plt.title("Trace.json")
plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1)); plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

---
## **课后实践**
请根据本节课程学习内容完成以下题目进行自测。

1. （单选题）在AscendC算子仿真调优环节，原文档中提到，开展算子性能仿真验证、获取性能数据的前置条件是？   
    A. 完成算子开发并能正常调用   
    B. 完成算子Kerne实现逻辑即可采集性能数据  
    C. 仅通过编译器静态分析代码逻辑，即可分析出性能数据，不需要借助工具   
    D. 手动计算理论性能值，不需要借助工具  

2. （单选题）Ascend C算子仿真数据抓取完成后，主要通过查看哪类核心文件来获取详细的性能耗时、单元利用率等关键性能数据？  
    A. 源码编译生成的.o目标文件  
    B. 仿真运行生成的性能统计trace.json和visualize_data.bin文件  
    C. 算子定义头文件  
    D. 系统通用日志文件  

3. （单选题）AscendC算子仿真性能分析时，重点关注的核心性能指标不包含以下哪一项？    
    A. 算子整体执行耗时  
    B. AI Core计算单元利用率  
    C. 数据搬运带宽与搬运耗时占比  
    D. 操作系统版本号  

4. （单选题）Ascend C算子仿真调优中的性能瓶颈定位，可行的方法是？  
    A. 随机修改代码尝试优化  
    B. 对比仿真报告中计算模块、搬运模块、同步等待的耗时占比，定位核心瓶颈  
    C. 仅查看代码行数判断效率  
    D. 依赖硬件自带的自动瓶颈提示  

5. （单选题）Ascend C算子仿真调优相比硬件实测，在算子性能优化阶段的核心优势是？  
    A. 无需依赖算子实际运行的实体昇腾硬件，借助仿真模拟实际硬件行为完成性能验证与瓶颈定位，缩短优化周期  
    B. 性能数据比硬件实测更精准  
    C. 可以直接修改硬件底层参数  
    D. 不需要编写任何算子代码  

执行以下代码查看答案：

In [ ]:
!cat ./answer/04.04_answer.txt